In [2]:
import sys
import os
import importlib

parent = os.path.dirname(os.getcwd())
sys.path.insert(0, parent)

from Helpers import loadData, multiplot, sphereMask, saveData, chunker, unchunker, kernelPad
importlib.reload(sys.modules['Helpers.chunker'])
importlib.reload(sys.modules['Helpers.unchunker'])
importlib.reload(sys.modules['Helpers.kernelPad'])
importlib.reload(sys.modules['Helpers'])

import h5py
import matplotlib.pyplot as plt
import numpy as np
import stackview
from scipy.ndimage import zoom

In [ ]:
path = r'C:\Users\Lab User\Desktop\ModernExperiments\exp_157\Scan_1.hdf5'
with h5py.File(path,"r") as f:
    print(list(f['RawData'].keys()))
    data = f['RawData']['Scan_1'][()]
    sz = np.shape(data)
    print(sz)

['Scan_1']
(4, 1216, 1024)


In [6]:
zData = zoom(data,.25)
print(np.shape(zData))

(1, 304, 256)


In [7]:
stackview.orthogonal(zData, continuous_update=True)

In [ ]:
edge_positions = np.argmax(data[:, :, :].max(axis=1), axis=1)
plt.plot(edge_positions)
plt.xlabel('Frame (Z)')
plt.ylabel('X position of bright edge')

In [ ]:
from skimage.registration import phase_cross_correlation

# Align each frame to the first frame
reference = data[0]
aligned = np.zeros_like(data)
aligned[0] = reference

for i in range(1, data.shape[0]):
    shift, _, _ = phase_cross_correlation(reference, data[i])
    aligned[i] = np.roll(data[i], shift.astype(int), axis=(0,1))